# Gao et al. (2026) on dandiset 001683: band power, asymmetry index and seizure PLI

Paper. Gao H, Frake A, Durand DM, He B (2026). Transcranial focused ultrasound stimulation
targeting white matter inhibits seizures in a rat model of epilepsy. *IEEE Trans Neural Syst Rehabil
Eng* 34, 251-259. [doi:10.1109/tnsre.2025.3644273](https://doi.org/10.1109/tnsre.2025.3644273). The
claims file was written from the NIH author manuscript (PMC12848950) and the supplement.

Dandiset. [DANDI:001683](https://dandiarchive.org/dandiset/001683/0.260109.2104), version
0.260109.2104, the only published version (the paper cites none; see `seizure_metrics.ipynb`).

This notebook streams all 70 labelled EEG blocks (2.9 GB each, about 205 GB) once and reduces each
as it loads (`tfus.process_block`): the Welch PSD of the two labelling channels (8 and 25, Fig. S1)
after a common average reference and a 60 Hz notch, over the whole block and over the seizure
periods; 30 to 80 Hz power in 1 s bins for the clock check of the labels; and the phase lag index
over the seizure periods for every pair of one ipsilateral and one contralateral channel. It takes
about 13 minutes with four processes.

| Claim | Tested here |
|---|---|
| fig4-no-theta-alpha-gamma-power-change | yes |
| fig4c-beta-decrease | yes |
| figs4-sham-power-no-change | yes |
| fig4ef-ai-increase | yes |
| figs5-seizures-lower-ai | blocked (data_missing): no pre-4-AP EEG in the dandiset; the tests of seizure AI against 0 are run |
| fig6-epileptic-pli-s1-decrease | yes |
| figs7-seizure-pli-s1-vs-non-s1 | blocked (data_missing): no pre-4-AP EEG in the dandiset |

In [ ]:
import time
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import stats

import tfus

warnings.filterwarnings("ignore")
pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 40)
pd.set_option("display.max_rows", 300)

started = time.time()
assets = tfus.list_assets()
big = assets[assets["size"] > 2e9]
lab = [m for m in tfus.read_all_meta(big.asset_id.tolist()) if m["kind"] == "Labeled_data"]
M = {m["asset_id"]: m for m in lab}
blocks = tfus.assign_blocks(lab)
print(f"dandiset {tfus.DANDISET_ID} version {tfus.DANDISET_VERSION}: {len(lab)} labelled blocks, "
      f"{sum(m['n_samples'] * m['n_channels'] * 8 for m in lab) / 1e9:.0f} GB of samples")

In [ ]:
t0 = time.time()
R = tfus.process_blocks(lab, workers=4)
print(f"processed {len(R)} blocks in {(time.time() - t0) / 60:.1f} min")

## Data check

Each check of the metadata against the values is stated under its output.

In [ ]:
print("Rats per group:", blocks.groupby("group").subject.nunique().to_dict(), "; blocks per group:", blocks.groupby("group").size().to_dict())
print("Blocks per rat:", blocks.groupby("subject").size().to_dict())
m0 = lab[0]
e = m0["electrodes"]
print("\nElectrode table of the EEG files (x, y in mm; location):")
print(e.assign(channel=np.arange(1, 33)).set_index("channel").T.to_string())
same = all(m["electrodes"][["x", "y"]].equals(m0["electrodes"][["x", "y"]]) for m in lab)
print("Same electrode table in every block file:", same)
print(f"Channel {tfus.IPSI_CH} at x={e.x[tfus.IPSI_CH - 1]}, y={e.y[tfus.IPSI_CH - 1]}; channel {tfus.CONTRA_CH} at x={e.x[tfus.CONTRA_CH - 1]}, y={e.y[tfus.CONTRA_CH - 1]}")
print("Burr holes (Methods A): AP -2 mm, ML +2.5 (right, injection, ipsilateral) and -2.5 (left)")
print(f"Channels 1-16 have x > 0: {bool((e.x[:16] > 0).all())}; channels 17-32 x < 0: {bool((e.x[16:] < 0).all())}")
plv = [m for m in tfus.read_all_meta(assets[(assets['size'] > 0.2e9) & (assets['size'] < 0.4e9)].asset_id.tolist()) if m["kind"].startswith("EEG data for PLV")]
print("\nNotes of the dandiset's PLV files (the source of the S1 channel lists):", sorted({m['notes'] for m in plv}))
f, p = R[m0["asset_id"]]["psd_block"]
print(f"\nPSD: {len(f)} bins, {f[1] - f[0]:.2f} Hz apart, 0 to {f[-1]:.0f} Hz; values {p.min():.2e} to {p.max():.2e} V^2/Hz")
raw = tfus.read_rows(m0, 1_000_000, 1_003_052)
print(f"Raw EEG, 1 s of block 1: {raw.min() * 1e6:.0f} to {raw.max() * 1e6:.0f} uV; per-channel SD {raw.std(axis=0).min() * 1e6:.0f} to {raw.std(axis=0).max() * 1e6:.0f} uV")

The electrode table puts channels 1 to 16 on the right (ipsilateral to the injection) and 17 to 32
on the left, and channels 8 and 25 at ML +/-2.8 mm, AP -3 mm, next to the burr holes, which fits
the encircled electrodes of Fig. S1. The values are in volts, tens to hundreds of microvolts, as an
EEG should be.

Clock check of the labels. The 30 to 80 Hz power of channel 8 (ipsilateral labels) and
channel 25 (contralateral labels) in 1 s bins, inside the labelled seizures against outside, with
the label times read as minutes, shifted later by 30 s to 600 s, and read as seconds.

In [ ]:
rows = []
for a, r in R.items():
    g = r["gamma_1s"]; nb = len(g)
    for k, h in enumerate(("ipsilateral", "contralateral")):
        s = tfus.seizures(M[a], h)
        for shift in (0, 30, 60, 120, 600, "read as seconds"):
            inside = np.zeros(nb, bool)
            pairs = s[["start", "stop"]].to_numpy() / (60 if shift == "read as seconds" else 1)
            for x, y in pairs:
                off = 0 if shift == "read as seconds" else shift
                inside[max(0, int(x + off)):min(nb, int(np.ceil(y + off)))] = True
            if inside.any() and (~inside).any():
                rows.append({"block": a, "hemi": h, "labels": shift, "inside/outside": g[inside, k].mean() / g[~inside, k].mean()})
cc = pd.DataFrame(rows)
print(cc.groupby("labels", sort=False)["inside/outside"].describe().round(2).to_string())
print(f"\nFile-hemispheres where gamma power is higher inside the labels (as minutes, no shift): "
      f"{(cc[cc.labels == 0]['inside/outside'] > 1).sum()} of {(cc.labels == 0).sum()}")

With the labels read as minutes, 30 to 80 Hz power inside them is many times that outside in every
block and hemisphere (the paper's labelling rule, Methods C); shifted by 60 s or more, or read as
seconds, the contrast is gone. So the labels and the EEG share a clock, and the label times are in
minutes. The two blocks without seizures (BH634 and BH638, post 2) have no labels to test.

In [ ]:
# Per rat, hemisphere, block: band power ratios (M5), under the readings segment (block, seizure)
# and total (3-300 Hz, 0 Hz to Nyquist); and the asymmetry index (Eq. 1) on the band power.
info = blocks.set_index("asset_id")
bp, ai = [], []
for a, r in R.items():
    i = info.loc[a]
    base = {"subject": i.subject, "group": i.group, "block": i.block}
    for seg, key in (("block", None), ("seizure", "seiz")):
        for k, h in enumerate(("ipsilateral", "contralateral")):
            if key is None:
                f, p = r["psd_block"]; p = p[:, k]
            else:
                f, p = r["psd_seiz_ipsi" if k == 0 else "psd_seiz_contra"]
                if f is None:
                    continue
                p = p[:, 0]
            pw = tfus.band_powers(f, p)
            for band in tfus.BANDS:
                for tot in ("total_3_300", "total_all"):
                    bp.append({**base, "hemi": h, "segment": seg, "total": tot, "band": band, "ratio": pw[band] / pw[tot]})
        f, p = r["psd_block"] if key is None else r["psd_seiz_merged"]
        if f is None:
            continue
        pw = tfus.band_powers(f, p)
        for band in tfus.BANDS:
            ip, co = pw[band]
            ai.append({**base, "segment": seg, "band": band, "ai": (ip - co) / (ip + co),
                       "ai_ratio": (ip / pw["total_3_300"][0] - co / pw["total_3_300"][1]) / (ip / pw["total_3_300"][0] + co / pw["total_3_300"][1])})
bp, ai = pd.DataFrame(bp), pd.DataFrame(ai)
print(f"Band power rows: {len(bp)}; AI rows: {len(ai)}")
print("Ranges of the band ratios:", bp.groupby(["band"]).ratio.agg(["min", "max"]).round(4).to_dict("index"))
print("Ranges of AI:", ai.groupby("band").ai.agg(["min", "max"]).round(3).to_dict("index"))
print("Blocks without a seizure PSD (no seizure of 2 s or more):", sorted(set((r.subject, r.block) for r in blocks.itertuples()) - set(map(tuple, bp[bp.segment == 'seizure'][["subject", "block"]].drop_duplicates().to_numpy()))))

## fig4-no-theta-alpha-gamma-power-change

> There are no significant changes in theta, alpha or gamma power ratios in either hemisphere.

A claim of absence. Band power ratios (M5) for sham and tFUS, blocks baseline to post 2, group by
block two-way ANOVA per hemisphere and band, with Bonferroni post hoc tests (M3). The claim covers
the within-group comparisons against baseline. Every reading of M5 (segment: whole block or
seizure periods; total: 3 to 300 Hz or 0 Hz to Nyquist) and of M3 is run. As in
`seizure_metrics.ipynb`, the repeated-measures reading is the one adopted where the ANOVA readings
disagree (see the summary there: it is the reading that gives the paper's own significant
within-group changes of Fig. 3).

In [ ]:
def band_tests(df, value="ratio", bands=tfus.BANDS, by_hemi=True):
    rows = []
    for (seg, tot), d in df.groupby(["segment", "total"]):
        for band in bands:
            for h in (("ipsilateral", "contralateral") if by_hemi else (None,)):
                dd = d[(d.band == band) & ((d.hemi == h) if h else True)]
                for k, (table, post) in tfus.group_block_tests(dd, value).items():
                    for r in post.itertuples():
                        rows.append({"segment": seg, "total": tot, "band": band, "hemi": h, "anova": k,
                                     "comparison": r.comparison, "family": r.family, "diff": r.diff,
                                     "p_family": r.p_family, "p_all": r.p_all})
    return pd.DataFrame(rows)

T4 = band_tests(bp)
w = T4[T4.family == "within"]
summary = (w.assign(sig_i=w.p_family < 0.05, sig_ii=w.p_all < 0.05)
             .groupby(["band", "segment", "total", "anova"])[["sig_i", "sig_ii"]].sum())
print("Number of significant within-group comparisons against baseline (of 12: 2 groups x 3 blocks x 2 hemispheres):")
print(summary.unstack("anova").to_string())
print("\nThe significant ones, theta, alpha and gamma:")
print(w[(w.band != "beta") & (w.p_family < 0.05)].round(4).to_string(index=False))

In [ ]:
print("Group mean band ratios (seizure segment, 3-300 Hz total):")
print(bp[(bp.segment == "seizure") & (bp.total == "total_3_300") & (bp.block != "post3")]
      .groupby(["band", "hemi", "group", "block"]).ratio.mean().unstack().round(4).to_string())
print("\nGroup mean band ratios (whole block, 3-300 Hz total):")
print(bp[(bp.segment == "block") & (bp.total == "total_3_300") & (bp.block != "post3")]
      .groupby(["band", "hemi", "group", "block"]).ratio.mean().unstack().round(4).to_string())
print("\nBetween-group comparisons (reported, not deciding), significant under the own family:")
b = T4[(T4.family == "between") & (T4.p_family < 0.05)]
print(b.round(4).to_string(index=False) if len(b) else "none")

**Result: not reproduced, resting on an assumption.** The ANOVA readings disagree, and the M5
readings agree within each. Under the ordinary reading no within-group comparison against baseline
is significant for theta, alpha or gamma under any of the four M5 readings. Under the
repeated-measures reading, adopted as the paper's test (see the summary), the sham theta ratio
rises above baseline in 4 or 5 of the 12 comparisons under every M5 reading (ipsilateral post 2:
p = 0.009 over the whole block, where the 3-300 Hz ratio goes from 0.098 to 0.136, and p = 0.0004
over the seizure periods), and the tFUS gamma ratio differs at ipsilateral post 2 with the 0 Hz to
Nyquist total (p = 0.043). Alpha has none. No between-group comparison is significant.

## fig4c-beta-decrease

> The beta power ratio of the tFUS group decreased significantly during stimulation and post 1
> compared with baseline.

Reproduced when, under a combination of the M3, M5 and hemisphere readings, the condition block
and post 1 are both significantly lower than baseline.

In [ ]:
wb = T4[(T4.band == "beta") & T4.comparison.isin(["tFUS baseline-condition", "tFUS baseline-post1"])]
print("tFUS beta ratio, baseline minus block (positive = decrease), with p under families (i) and (ii):")
print(wb.pivot_table(index=["segment", "total", "anova", "hemi"], columns="comparison", values=["diff", "p_family", "p_all"]).round(4).to_string())
both = wb.assign(ok=(wb.p_family < 0.05) & (wb["diff"] > 0)).groupby(["segment", "total", "anova", "hemi"]).ok.sum()
print("\nCombinations with both comparisons significant decreases (family i):", list(both[both == 2].index))
print("\ntFUS beta ratio means:")
print(bp[(bp.band == "beta") & (bp.group == "tFUS") & (bp.block != "post3")].groupby(["segment", "total", "hemi", "block"]).ratio.mean().unstack().round(4).to_string())

**Result: partially reproduced.** Over the whole block under the repeated-measures reading the
tFUS beta ratio at post 1 is significantly below baseline in both hemispheres (3-300 Hz total:
p = 0.0028 ipsilateral, 0.026 contralateral; 0 Hz to Nyquist: 0.0005 and 0.0094), but the condition
block is not (p = 0.16 to 0.60), although its mean is lower (ipsilateral 0.445 at baseline, 0.417
at the condition block). Over the seizure periods the beta ratio does not fall (ipsilateral 0.493,
0.487 and 0.500) and no comparison is significant. Under the ordinary reading none is.

## figs4-sham-power-no-change

> In the sham group, band power ratios show no significant change among time blocks or between
> hemispheres.

A claim of absence. Sham rats, five blocks, per band a two-way ANOVA (hemisphere by block) with
Bonferroni post hoc tests; no block or hemisphere effect nor post hoc comparison may be significant
in any band.

In [ ]:
rows = []
for (seg, tot), d in bp[bp.group == "sham"].groupby(["segment", "total"]):
    for band in tfus.BANDS:
        dd = d[d.band == band]
        for k, (table, post) in tfus.hemi_block_tests(dd, "ratio").items():
            rows.append({"segment": seg, "total": tot, "band": band, "anova": k,
                         "p_hemi": table.p.iloc[0], "p_block": table.p.iloc[1], "p_interaction": table.p.iloc[2],
                         "posthoc_sig_i": int((post.p_family < 0.05).sum()), "posthoc_sig_ii": int((post.p_all < 0.05).sum()),
                         "smallest": post.loc[post.p_family.idxmin(), "comparison"], "smallest_p": post.p_family.min()})
S4 = pd.DataFrame(rows)
print(S4.round(4).to_string(index=False))

**Result: not reproduced, resting on an assumption.** Beta varies with block and hemisphere: over
the whole block the block and hemisphere effects are significant under both readings (ordinary
block p = 0.0004 and 0.0002 for the two totals, hemisphere p below 0.0001, with 4 and 5 post hoc
comparisons significant), and over the seizure periods under the repeated-measures reading (block
p = 0.0009 and 0.0007, 2 post hoc comparisons). Gamma has a hemisphere effect over the whole block
(ordinary p = 0.0037 and 0.0004). Under the ordinary reading over the seizure periods nothing is
significant in any band (smallest p = 0.09), so the M5 readings disagree under that ANOVA reading;
under the repeated-measures reading, the one adopted, they agree.

## fig4ef-ai-increase

> The normalized asymmetry index of the tFUS group increases significantly relative to baseline in
> the theta and alpha bands, and differs significantly from the sham group in every band.

AI per rat, band and block (Eq. 1, on the band power of channels 8 and 25), normalized to the
group's mean baseline AI by (a) division or (b) subtraction (M5), for the whole block and for the
seizure periods (the union of both hemispheres' labels); group by block two-way ANOVA per band.
Reproduced when, under one combination of readings, at least one tFUS block is significantly above
baseline in theta and in alpha, and sham against tFUS is significant at one block or more in each
of the four bands.

In [ ]:
d = ai[ai.block != "post3"].copy()
base = d[d.block == "baseline"].groupby(["segment", "band", "group"]).ai.mean().rename("base")
d = d.join(base, on=["segment", "band", "group"])
d["norm_div"] = d.ai / d.base
d["norm_sub"] = d.ai - d.base
print("Group mean baseline AI (its sign decides what an increase means under division):")
print(base.unstack().round(4).to_string())
rows = []
for seg in ("block", "seizure"):
    for norm in ("norm_div", "norm_sub"):
        for band in tfus.BANDS:
            dd = d[(d.segment == seg) & (d.band == band)]
            for k, (table, post) in tfus.group_block_tests(dd, norm).items():
                for r in post.itertuples():
                    rows.append({"segment": seg, "norm": norm, "band": band, "anova": k, "comparison": r.comparison,
                                 "family": r.family, "diff": r.diff, "p_family": r.p_family, "p_all": r.p_all})
TA = pd.DataFrame(rows)
print("\nGroup means of normalized AI:")
print(d.groupby(["segment", "band", "group", "block"])[["norm_div", "norm_sub"]].mean().unstack().round(3).to_string())

In [ ]:
def ai_verdict(t):
    up = {}
    for band in ("theta", "alpha"):
        x = t[(t.band == band) & t.comparison.str.startswith("tFUS baseline")]
        up[band] = bool(((x.p_family < 0.05) & (x["diff"] < 0)).any())       # baseline minus block < 0: increase
    between = {band: bool((t[(t.band == band) & (t.family == "between")].p_family < 0.05).any()) for band in tfus.BANDS}
    return pd.Series({**{f"tFUS up {k}": v for k, v in up.items()}, **{f"between {k}": v for k, v in between.items()}})

V = TA.groupby(["segment", "norm", "anova"]).apply(ai_verdict)
print(V.to_string())
print("\nSignificant comparisons (own family):")
print(TA[TA.p_family < 0.05].round(4).to_string(index=False))

**Result: partially reproduced.** The mean baseline AI is negative in both groups (whole block:
sham -0.58 to -0.61, tFUS -0.24 to -0.29), so the two normalizations read an increase in opposite
directions. With subtraction (reading b), the whole block and the repeated-measures reading, the
normalized AI of the tFUS group rises above baseline, toward balance, in theta (post 1 p = 0.033,
post 2 p = 0.0047) and alpha (post 2 p = 0.0057). Sham against tFUS is not significant in any
band, block or combination of readings. With division the same change appears as a significant
fall (theta post 2 p = 0.0006), and over the seizure periods nothing is significant.

## figs5-seizures-lower-ai

> During seizures (baseline block, after 4-AP) the asymmetry index is negative and differs from
> zero in every band, more than in the pre-4-AP baseline state; seizures disrupt the normal
> interhemispheric balance.

The pre-4-AP baseline state is not in the dandiset: the resting recordings of the target
confirmation (`EP data`, `BASELINE`) hold only the 64 intracranial channels of left S1, not the
EEG, and no EEG recording before the first injection was deposited (the labelled blocks start
with the baseline block, after the monitoring block). The claim is blocked; the part that can be
run, seizure AI against 0 (one-sample t-tests, Fig. S5 legend) for the 16 rats and for the 10 tFUS
rats, is shown.

In [ ]:
sa = ai[(ai.segment == "seizure") & (ai.block == "baseline")]
rows = []
for rats, dd in (("16 rats", sa), ("10 tFUS rats", sa[sa.group == "tFUS"])):
    for band in tfus.BANDS:
        x = dd[dd.band == band].ai
        t = stats.ttest_1samp(x, 0)
        rows.append({"rats": rats, "band": band, "n": len(x), "mean AI": x.mean(), "sem": x.sem(), "t": t.statistic, "p": t.pvalue})
print(pd.DataFrame(rows).round(4).to_string(index=False))

**Result: blocked (data_missing).** The pre-4-AP state the claim compares with is not in the
dandiset. The part that can be run holds: the seizure AI of the baseline block is negative in every
band, -0.41 to -0.43 over the 16 rats (p < 0.0001 in each band) and -0.27 to -0.30 over the 10
tFUS rats (p = 0.0001 to 0.0012).

## fig6-epileptic-pli-s1-decrease

> In epileptic rats, the S1 PLI of the tFUS group decreases in theta, alpha, beta and gamma bands
> during stimulation, post 1 and post 2 relative to baseline seizures, while non-S1 PLI does not
> decrease; the decrease of S1 connectivity is significant.

PLI (M6) over the seizure periods of each block (the union of both hemispheres' labels, periods of
1 s or more), per band, averaged over the 16 S1 pairs (left S1 with right S1 channels) and the 144
non-S1 pairs; per seizure and averaged (reading a) or over the concatenated seizure samples
(reading b); normalized to the group's mean baseline-block PLI by difference or ratio. Reproduced
when, under one combination of readings, the tFUS S1 mean is below baseline in all four bands for
the condition block, post 1 and post 2, the tFUS non-S1 mean is not below baseline in most of
those 12 cells, and a group by block two-way ANOVA per band and region finds sham against tFUS
significant for S1 in at least one band and block.

In [ ]:
rows = []
for a, r in R.items():
    i = info.loc[a]
    for band in tfus.BANDS:
        v = r["pli"][band]
        if v is None:
            continue
        for reading in ("per_seizure", "concat"):
            pm = tfus.pair_means(v[reading])
            for region in ("S1", "nonS1"):
                rows.append({"subject": i.subject, "group": i.group, "block": i.block, "band": band,
                             "reading": reading, "region": region, "pli": pm[region], "n_seizures": v["n"], "seconds": v["seconds"]})
P = pd.DataFrame(rows)
P = P[P.block != "post3"]
print("Rat-blocks with seizure PLI:", P[["subject", "block"]].drop_duplicates().shape[0], "of", int((blocks.block != "post3").sum()))
print("PLI range:", round(P.pli.min(), 3), "to", round(P.pli.max(), 3))
base = P[P.block == "baseline"].groupby(["group", "band", "reading", "region"]).pli.mean().rename("base")
P = P.join(base, on=["group", "band", "reading", "region"])
P["diff"] = P.pli - P.base
P["ratio"] = P.pli / P.base
G = P.groupby(["reading", "region", "band", "group", "block"])[["diff", "ratio"]].mean()
print("\nGroup mean normalized PLI (difference from the group's baseline):")
print(G["diff"].unstack().round(4).to_string())

In [ ]:
rows = []
for reading in ("per_seizure", "concat"):
    for norm in ("diff", "ratio"):
        g = G[norm].xs(reading, level="reading").unstack()
        ref = 0.0 if norm == "diff" else 1.0
        t = g.xs("tFUS", level="group")
        s1 = t.xs("S1", level="region")[["condition", "post1", "post2"]]
        ns = t.xs("nonS1", level="region")[["condition", "post1", "post2"]]
        sig = []
        for band in tfus.BANDS:
            dd = P[(P.reading == reading) & (P.region == "S1") & (P.band == band)]
            for k, (table, post) in tfus.group_block_tests(dd, norm).items():
                for r in post[(post.family == "between") & (post.p_family < 0.05)].itertuples():
                    sig.append(f"{band} {r.comparison} [{k}] p={r.p_family:.3g}")
        rows.append({"reading": reading, "norm": norm, "tFUS S1 cells below baseline (of 12)": int((s1 < ref).sum().sum()),
                     "tFUS non-S1 cells not below baseline (of 12)": int((ns >= ref).sum().sum()),
                     "S1 sham-tFUS significant": "; ".join(sig) if sig else "none"})
print(pd.DataFrame(rows).to_string(index=False))
print("\nPer rat, tFUS S1 PLI difference from the group baseline, per_seizure reading:")
print(P[(P.group == "tFUS") & (P.region == "S1") & (P.reading == "per_seizure")].pivot_table(index=["band", "subject"], columns="block", values="diff").round(3).unstack(0).to_string())

**Result: partially reproduced.** Per seizure and averaged (reading a of M6), the tFUS S1 PLI is
below the group's baseline in 8 of the 12 band and block cells: in theta and alpha at all three
blocks (alpha -0.026, -0.032 and -0.057), in beta only at the condition block and in gamma only at
post 1; over the concatenated seizure samples, 6 of 12. The non-S1 PLI falls too: it is not below
baseline in only 4 of 12 cells, so that part of the claim fails. No sham against tFUS comparison of
S1 PLI is significant in any band or block under either ANOVA reading. The criterion's partial
outcome (S1 decreases in some bands only, without a significant group difference) is the one that
applies; the S1 specificity the claim describes is not seen, and the sham S1 PLI falls as much in
theta (-0.022 to -0.080).

## figs7-seizure-pli-s1-vs-non-s1

> Comparing seizures (baseline block) with the pre-4-AP baseline state, the PLI change of S1 pairs
> differs significantly from that of non-S1 pairs in every band.

**Result: blocked (data_missing).** The change is from the pre-4-AP baseline state, whose EEG is
not in the dandiset (see figs5-seizures-lower-ai).

## Summary

A failure to reproduce here may be a failure of this attempt, not of the paper: see the reasoning
and the differences from the paper's method.

| Claim | Outcome |
|---|---|
| fig4-no-theta-alpha-gamma-power-change | not reproduced (assumed) |
| fig4c-beta-decrease | partially reproduced |
| figs4-sham-power-no-change | not reproduced (assumed) |
| fig4ef-ai-increase | partially reproduced |
| figs5-seizures-lower-ai | blocked: data missing |
| fig6-epileptic-pli-s1-decrease | partially reproduced |
| figs7-seizure-pli-s1-vs-non-s1 | blocked: data missing |

The outcomes of the other fifteen claims are in `seizure_metrics.ipynb` and `wild_type.ipynb`.

How the analysis differs from the paper's. No ICA is applied: which components the authors
removed is not recorded. The EEG is re-referenced to the common average and notched at 60 Hz, as
in Methods C. The labelling channels 8 and 25 are read from the drawing of Fig. S1 and the S1
channels from the notes of the dandiset's PLV files; the paper's text lists neither. The segment
analysed, the total of the band ratio and the normalization of AI and PLI are not fixed by the
paper and are run under each reading. The repeated-measures ANOVA reading is adopted where the
ANOVA readings disagree on a claim of absence, as in `seizure_metrics.ipynb`. During the condition
block of the tFUS rats the EEG carries the pulse responses at 1 Hz; the paper does not say whether
they were removed, and they are not.

What was found about the dataset. The seizure labels share the EEG's clock and are in minutes:
30 to 80 Hz power inside them is 2.5 to 50 times that outside, and the contrast is gone when they
are shifted by a minute or read as seconds. The pre-4-AP EEG of Figs. S5 and S7 is not in the
dandiset. The baseline AI is strongly negative in both groups (mean contralateral band power about 1.7
(tFUS) to 4 (sham) times the ipsilateral), which the paper does not report.

In [ ]:
print(f"Notebook ran in {(time.time() - started) / 60:.1f} min")